# Apprentissage de représentation, texte et recommandation

## Partie D : représentations de mots, fine-tuning et convolutions sur le texte

Vincent Guigue (vincent.guigue@agroparistech.fr)

[directement inspiré de :] Nicolas Baskiotis (nicolas.baskiotis@sorbonne-universite.fr), Benjamin Piwowarski (benjamin.piwowarski@sorbonne-universite.fr) -- MLIA/ISIR, Sorbonne Université

**Plan :** outils de base pour le texte (tokenisation, vocabulaire, padding) $\to$ classification de sentiment avec un modèle « moyen » appris de bout en bout $\to$ représentations pré-apprises (fasttext), gelées ou *fine-tunées* $\to$ réseau convolutionnel sur le texte et analyse de ce que détectent ses filtres.

**Budget :** chaque apprentissage dure de 30 s à 2 min environ (cf. commentaires de la configuration). Les modèles appris sont sauvegardés dans `models/` et rechargés automatiquement (variable `TRAIN`).

## Table des matières

- [0. Configuration (device, tensorboard, sauvegardes)](#sec-1)
- [1. Outils de base pour le texte](#sec-2)
- [2. Données IMDB : DataLoader et padding](#sec-3)
- [3. Exercice : apprentissage de bout en bout (*end-to-end*)](#sec-4)
  - [3.1 Évaluation et visualisation des embeddings de mots appris](#sec-5)
- [4. Représentations pré-apprises et fine-tuning](#sec-6)
- [5. Convolutions et texte](#sec-7)
  - [5.1 Portée (champ réceptif) d'une sortie](#sec-8)
  - [5.2 Le modèle convolutionnel](#sec-9)
- [6. Que détectent les filtres du CNN ?](#sec-10)

# <a id="sec-1"></a>0. Configuration (device, tensorboard, sauvegardes)

In [1]:
import heapq
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_dataset     # bibliothèque HuggingFace (pip install datasets)

# Sélection du device : GPU NVIDIA (cuda) > GPU Apple (mps) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
# device = torch.device("cpu")

# Sur Mac (mps), le modèle moyen (parties 3-4) est BEAUCOUP plus rapide sur CPU (~5 s/époque contre 30 s à plusieurs
# minutes, surtout avec fine-tuning), alors que le CNN (partie 5) est plus rapide sur mps (~20 s/époque contre ~85 s)
device_moyen = torch.device("cpu") if device.type == "mps" else device

torch.manual_seed(0)
print("device :", device, "| modèle moyen :", device_moyen)

# Sauvegarde des modèles : False => on recharge les modèles de models/ s'ils existent (sinon on apprend)
#                          True  => on ré-apprend tout (et on écrase les sauvegardes)
TRAIN = False
MODELS = Path("models")
MODELS.mkdir(exist_ok=True)

device : mps | modèle moyen : cpu


In [2]:
from torch.utils.tensorboard import SummaryWriter

import tempfile
TB_PATH = Path(tempfile.gettempdir()) / "logs" / "module3"
TB_PATH.mkdir(parents=True, exist_ok=True)
TB_INLINE = True   # True : tensorboard affiché dans le notebook ; False : le lancer soi-même dans un terminal
print(f"Pour lancer tensorboard dans un terminal : tensorboard --logdir {TB_PATH}")
if TB_INLINE:
    %load_ext tensorboard
    %tensorboard --logdir {TB_PATH}

Pour lancer tensorboard dans un terminal : tensorboard --logdir C:\Users\THEOPO~1\AppData\Local\Temp\logs\module3


# <a id="sec-2"></a>1. Outils de base pour le texte

L'utilisation la plus courante de l'apprentissage de représentation est de fournir une représentation utile pour la classification. Dans ce notebook, nous utiliserons le corpus de critiques IMDB pour l'analyse de sentiment : un corpus qui contient des critiques de films, dont l'objectif est de prédire si une critique est positive ou négative.

Avant de travailler sur IMDB, trois outils, illustrés sur un exemple jouet :
1. la **tokenisation** : découper un texte en une liste de *tokens* (mots, ponctuation),
2. le **vocabulaire** : associer un indice à chaque token (ce sont ces indices que l'on donne à un `nn.Embedding`), avec deux mots spéciaux : `<pad>` (remplissage, cf. 4.) et `<unk>` (mot inconnu ou trop rare),
3. le **padding** : compléter les phrases pour qu'elles aient toutes la même longueur, et pouvoir les empiler dans un tenseur.

*Note :* ces outils étaient fournis par la bibliothèque `torchtext`, qui n'est plus maintenue depuis 2024 (incompatible avec les versions récentes de pytorch). Ils s'écrivent en quelques lignes ; pour des usages avancés, on utilise les *tokenizers* de HuggingFace (cf. module Transformers).

In [3]:
def tokenizer(text):
    """minuscules + découpage en mots / signes de ponctuation (les balises <br /> d'IMDB sont supprimées)"""
    return re.findall(r"\w+|[^\w\s]", text.lower().replace("<br />", " "))


class Vocab:
    """Correspondance token <-> indice. Les tokens vus moins de min_freq fois sont remplacés par <unk>.
    Les indices sont attribués par fréquence décroissante (après les mots spéciaux)."""
    def __init__(self, token_lists, min_freq=1, specials=("<pad>", "<unk>")):
        counts = Counter(t for tokens in token_lists for t in tokens)
        words = sorted((w for w, c in counts.items() if c >= min_freq), key=lambda w: (-counts[w], w))
        self.itos = list(specials) + words                      # indice -> token
        self.stoi = {w: i for i, w in enumerate(self.itos)}     # token -> indice
        self.unk = self.stoi["<unk>"]

    def __len__(self):
        return len(self.itos)

    def __contains__(self, token):
        return token in self.stoi

    def __getitem__(self, token):
        return self.stoi.get(token, self.unk)

    def encode(self, tokens):
        return [self[t] for t in tokens]

    def decode(self, indices):
        return [self.itos[int(i)] for i in indices]

In [4]:
# test 1 : tokenisation des phrases
phrs = ["the cat sat on the mat.", "the dog eats", "The cat and the dog!"]
phrs_tok = [tokenizer(t) for t in phrs]
print(phrs_tok)

# test 2 : indexation des mots (avec les mots spéciaux)
toy_vocab = Vocab(phrs_tok)
print("Dictionnaire :", toy_vocab.itos, "| taille :", len(toy_vocab))
print("indice de 'cat' :", toy_vocab["cat"], "| d'un mot inconnu :", toy_vocab["elephant"])

# test 3 : indexation des phrases, et transformation inverse
phr_ind = [torch.tensor(toy_vocab.encode(phr)) for phr in phrs_tok]
print("Phrases indexées :", phr_ind)
print("Reconstruction :", toy_vocab.decode(phr_ind[0]))

# test 4 : padding => toutes les phrases ont la même longueur
phr_ind_pad = pad_sequence(phr_ind, batch_first=True, padding_value=toy_vocab["<pad>"])
print("Phrases paddées :\n", phr_ind_pad)
print("Reconstruction (pad) :", toy_vocab.decode(phr_ind_pad[1]))

[['the', 'cat', 'sat', 'on', 'the', 'mat', '.'], ['the', 'dog', 'eats'], ['the', 'cat', 'and', 'the', 'dog', '!']]
Dictionnaire : ['<pad>', '<unk>', 'the', 'cat', 'dog', '!', '.', 'and', 'eats', 'mat', 'on', 'sat'] | taille : 12
indice de 'cat' : 3 | d'un mot inconnu : 1
Phrases indexées : [tensor([ 2,  3, 11, 10,  2,  9,  6]), tensor([2, 4, 8]), tensor([2, 3, 7, 2, 4, 5])]
Reconstruction : ['the', 'cat', 'sat', 'on', 'the', 'mat', '.']
Phrases paddées :
 tensor([[ 2,  3, 11, 10,  2,  9,  6],
        [ 2,  4,  8,  0,  0,  0,  0],
        [ 2,  3,  7,  2,  4,  5,  0]])
Reconstruction (pad) : ['the', 'dog', 'eats', '<pad>', '<pad>', '<pad>', '<pad>']


# <a id="sec-3"></a>2. Données IMDB : DataLoader et padding

Les données sont récupérées avec la bibliothèque [`datasets`](https://huggingface.co/docs/datasets) de HuggingFace (téléchargement d'environ 80 Mo la première fois, puis mise en cache) : 25 000 critiques en apprentissage, 25 000 en test, étiquettes 0 = négatif, 1 = positif.

On tokenise tous les textes **une seule fois** et on construit le vocabulaire sur le jeu d'apprentissage (tokens vus au moins 20 fois).

In [5]:
imdb = load_dataset("stanfordnlp/imdb")
print(imdb["train"][0]["text"][:300], "...\nlabel :", imdb["train"][0]["label"])

train_tok = [tokenizer(t) for t in tqdm(imdb["train"]["text"], desc="tokenisation train")]
test_tok = [tokenizer(t) for t in tqdm(imdb["test"]["text"], desc="tokenisation test")]

imdb_vocab = Vocab(train_tok, min_freq=20)
PAD = imdb_vocab["<pad>"]
print("Taille du vocabulaire :", len(imdb_vocab))
print(imdb_vocab.itos[:20])     # les 20 premiers tokens (mots spéciaux puis les plus fréquents)
print(imdb_vocab["the"])        # indice d'un token

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...
label : 0


Taille du vocabulaire : 13395
['<pad>', '<unk>', 'the', '.', ',', 'and', 'a', 'of', 'to', "'", 'is', 'it', 'in', 'i', 'this', 'that', '-', '"', 's', 'was']
2


**Longueur des textes.** Les critiques ont des longueurs très variables. Pour borner le temps de calcul (en particulier pour le CNN de la partie 5), on **tronque** chaque critique à ses `MAX_LEN` premiers tokens : cela ne concerne qu'environ une critique sur cinq, et coûte très peu en performance (le début d'une critique est généralement aussi informatif que la fin).

In [6]:
lengths = np.array([len(t) for t in train_tok])
print(f"longueur moyenne {lengths.mean():.0f}, médiane {np.median(lengths):.0f}, "
      f"90e centile {np.percentile(lengths, 90):.0f}, max {lengths.max()}")

MAX_LEN = 400
print(f"proportion de critiques tronquées : {(lengths > MAX_LEN).mean():.1%}")

def encode(tokens):
    return torch.tensor(imdb_vocab.encode(tokens[:MAX_LEN]))

# jeux de données = listes de couples (tenseur d'indices, étiquette)
imdb_train = list(zip([encode(t) for t in train_tok], imdb["train"]["label"]))
imdb_test = list(zip([encode(t) for t in test_tok], imdb["test"]["label"]))
print(imdb_train[0][0][:20], imdb_train[0][1])

longueur moyenne 279, médiane 208, 90e centile 547, max 2803
proportion de critiques tronquées : 18.7%


tensor([  13, 1632,   13,  258, 2017,   16, 4162,   46,   69,  384, 1127,   96,
           7,   39,    2, 7147,   15, 3435,   11,   63]) 0


Dans un corpus de texte, les phrases n'ont généralement pas toutes la même longueur. Cela pose problème pour la constitution des mini-batchs sous forme de tenseurs, et en général pour toute opération tensorielle dans le réseau. Une solution simple consiste à faire du *padding*, c'est-à-dire rajouter à la fin des phrases un token spécial `<pad>` pour les compléter jusqu'à la longueur voulue. Ainsi les mini-batchs sont constitués de phrases de même longueur, mais qui contiennent le token `<pad>`. On utilise généralement un vecteur nul pour sa représentation (et l'indice 0). Il faut alors veiller à ce que ce token ne perturbe pas les calculs (cf. la moyenne de la partie 3) ; pour les tâches où l'on prédit un token par position, on « masque » les erreurs dues à ce token lors du calcul du coût (argument `ignore_index` de `nn.CrossEntropyLoss`).

Il faut donc un **DataLoader** spécifique pour un tel corpus, en charge du padding. L'argument `collate_fn(batch)` d'un `DataLoader` permet de spécifier une fonction qui agrège la liste `batch` des exemples individuels tirés dans le jeu de données, et renvoie le mini-batch associé.

**TODO :** écrire la fonction `collate_batch_imdb(batch)` : `batch` est une liste de couples `(tenseur d'indices, étiquette)` ; elle doit renvoyer le couple *(tenseur paddé de taille batch x longueur, tenseur des étiquettes)*. Utiliser [`pad_sequence(l, batch_first=True, padding_value=PAD)`](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html).

In [ ]:
def collate_batch_imdb(batch):
    texts, labels = zip(*batch)       # cf. notebook 3_0 : séparer une liste de couples en deux tuples
    ##  TODO 

imdb_train_dataloader = DataLoader(imdb_train, batch_size=32, shuffle=True, collate_fn=collate_batch_imdb)
imdb_test_dataloader = DataLoader(imdb_test, batch_size=256, shuffle=False, collate_fn=collate_batch_imdb)

# vérification du résultat
x, y = next(iter(imdb_train_dataloader))
print(x.shape, y.shape)
print(x[:3, -10:], y[:3])

# <a id="sec-4"></a>3. <span class="alert-success">Exercice : apprentissage de bout en bout (*end-to-end*)</span>

Le premier réseau que nous allons étudier est composé d'une couche d'embeddings pour représenter les tokens, puis de plusieurs couches linéaires avec des ReLU comme fonction d'activation. La représentation d'une critique est la **moyenne des représentations de ses tokens**. Le réseau est appris de bout en bout : les embeddings des mots sont appris en même temps que le classifieur, pour la tâche de classification.

Codez le réseau `ModeleMoyen` : n'oubliez pas de prendre en compte le token `<pad>` dans la construction de la représentation de la phrase (la moyenne ne doit porter que sur les vrais mots). En vrai, cela n'a pas beaucoup d'importance dans ce cas précis : voyez-vous pourquoi ?

Vous pouvez utiliser 100 dimensions pour la représentation, et deux couches cachées de 50 et de 20 neurones suivies de ReLU.

<img src="./media/modaverage.png">

Entraînez le réseau, visualisez les embeddings obtenus.

*Note :* les poids fournis dans `models/` supposent la même structure que la correction : `self.embedding`, puis `self.fc = nn.Sequential(Linear, ReLU, Linear, ReLU, Linear)`.

In [ ]:
EPOCHS = 3    # suffisant : au-delà, tous les modèles de ce notebook sur-apprennent (cf. courbes de test)

def accuracy(yhat, y):
    return (yhat.argmax(1) == y).float().mean().item()


class ModeleMoyen(nn.Module):
    def __init__(self, vocab_size, input_dim, output_dim, layers, padding_idx, pre_trained=None, freeze=True, name="moyen"):
        """
        vocab_size : taille du vocabulaire
        input_dim : taille des embeddings
        output_dim : nb de classes (ATTENTION, même dans le cas binaire, mettre 2 avec la loss CrossEntropy)
        layers : liste des dimensions des couches cachées [h_layer1, h_layer2, ...]
        padding_idx : indice du token de padding (embedding nul et non appris)
        pre_trained : tenseur (vocab_size, input_dim) d'embeddings pré-appris (partie 4), ou None
        freeze : si pre_trained est fourni, les embeddings sont-ils gelés (True) ou fine-tunés (False) ?
        """
        super().__init__()
        self.name = name
        self.padding_idx = padding_idx
        # 1) la table des représentations
        if pre_trained is None:
            self.embedding = nn.Embedding(vocab_size, input_dim, padding_idx=padding_idx)
        else:
            # clone() : from_pretrained ne copie PAS le tenseur ; sans clone, le fine-tuning modifierait en place
            # pre_trained (fast_imdb_vectors), et donc les embeddings de tous les modèles qui le partagent !
            self.embedding = nn.Embedding.from_pretrained(pre_trained.clone(), freeze=freeze, padding_idx=padding_idx)
        # 2) construire les couches Linear + ReLU (une par élément de layers), puis la couche de sortie
        # 3) les mettre dans un Sequential self.fc
        #       note : si listecouches = [couche1, couche2, ...] => self.fc = nn.Sequential(*listecouches)
        ##  TODO 

    def forward(self, x):
        # 1) récupération des embeddings des x
        # 2) moyenne = somme / nombre de vrais mots (pas les pads)
        # 3) passage du résultat dans self.fc
        ##  TODO 


def evaluate(model, loader, loss_fn=nn.CrossEntropyLoss()):
    """renvoie (loss moyenne, accuracy) sur loader"""
    device = next(model.parameters()).device     # on calcule là où se trouve le modèle
    model.eval()
    cumloss, cumacc, count = 0., 0., 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            yhat = model(x)
            cumloss += loss_fn(yhat, y).item() * len(x)
            cumacc += accuracy(yhat, y) * len(x)
            count += len(x)
    return cumloss / count, cumacc / count


# tjs le même code (ou presque)
def train(model, epochs, train_loader, test_loader, lr=1e-3, device=device):
    writer = SummaryWriter(TB_PATH / f"{model.name}-{time.strftime('%H%M%S')}")
    model = model.to(device)
    # on ne donne à l'optimiseur que les paramètres à apprendre (pas les embeddings gelés)
    optim = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    loss_fn = nn.CrossEntropyLoss()   # ATTENTION : 2 sorties pour un problème binaire avec la cross-entropie
    for epoch in range(epochs):
        model.train()
        cumloss, cumacc, count = 0., 0., 0
        for x, y in tqdm(train_loader, desc=f"{model.name} {epoch + 1}/{epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            optim.zero_grad()
            yhat = model(x)
            l = loss_fn(yhat, y)
            l.backward()
            optim.step()
            cumloss += l.item() * len(x)    # .item() : sinon on garde tout le graphe de calcul en mémoire !
            cumacc += accuracy(yhat, y) * len(x)
            count += len(x)
        # on suit les courbes sur le test (pas de sélection de modèle ici, donc pas besoin de validation)
        test_loss, test_acc = evaluate(model, test_loader, loss_fn)
        writer.add_scalar("loss/train", cumloss / count, epoch)
        writer.add_scalar("accuracy/train", cumacc / count, epoch)
        writer.add_scalar("loss/test", test_loss, epoch)
        writer.add_scalar("accuracy/test", test_acc, epoch)
        print(f"[{model.name}] epoch {epoch + 1} | train loss {cumloss / count:.3f} acc {cumacc / count:.3f}"
              f" | test loss {test_loss:.3f} acc {test_acc:.3f}")
    writer.close()
    return model


def train_or_load(model, epochs, fichier, retrain=None, device=device):
    """apprend le modèle et le sauvegarde dans models/fichier, ou le recharge s'il existe déjà (et TRAIN=False)
    les embeddings gelés ne sont pas sauvegardés (ils ne changent pas : on les recharge depuis fasttext)"""
    fichier = MODELS / fichier
    retrain = TRAIN if retrain is None else retrain
    frozen = {k for k, p in model.named_parameters() if not p.requires_grad}
    if retrain or not fichier.exists():
        train(model, epochs, imdb_train_dataloader, imdb_test_dataloader, device=device)
        torch.save({k: v for k, v in model.state_dict().items() if k not in frozen}, fichier)
    else:
        missing, unexpected = model.load_state_dict(torch.load(fichier, map_location="cpu"), strict=False)
        assert set(missing) <= frozen and not unexpected, f"structure incompatible : {missing} {unexpected}"
        model.to(device)
        print(f"[{model.name}] chargé depuis {fichier} | test acc {evaluate(model, imdb_test_dataloader)[1]:.3f}")
    return model

In [ ]:
## Création et entraînement du modèle moyen
# 1) trouver les bons arguments pour construire votre réseau
# 2) lancer l'apprentissage (train_or_load)

##  TODO 

In [ ]:
# Pourquoi la prise en compte du <pad> n'a-t-elle pas beaucoup d'importance ici ?
##  TODO 

## <a id="sec-5"></a>3.1 Évaluation et visualisation des embeddings de mots appris

- les courbes d'apprentissage (et de test) doivent être très bonnes (~85 % de bonne classification) ;
- les représentations de mots ont l'air d'avoir un peu de sens... mais ce n'est pas très impressionnant.

In [11]:
MIN_LEN = 32    # longueur minimale (le CNN de la partie 5 a besoin d'un texte au moins aussi long que son champ réceptif)

def predict(model, text):
    """probabilité que le texte soit une critique positive"""
    ids = encode(tokenizer(text))
    x = F.pad(ids, (0, max(0, MIN_LEN - len(ids))), value=PAD).unsqueeze(0).to(next(model.parameters()).device)
    model.eval()
    with torch.no_grad():
        return model(x).softmax(-1)[0, 1].item()

# test d'exemples
for phr in ["this movie is awful", "the movie is great", "not bad at all", "I expected a great movie, it was boring"]:
    print(f"{predict(modeleMoyen, phr):.2f}  {phr}")

0.00  this movie is awful
1.00  the movie is great
0.00  not bad at all
0.01  I expected a great movie, it was boring


In [12]:
# recherche de la première erreur dans le test
for k, (x, label) in enumerate(imdb_test):
    pred = int(predict(modeleMoyen, imdb["test"][k]["text"]) > 0.5)
    if pred != label:
        print(f"critique {k} : vraie étiquette {label}, prédiction {pred}\n")
        print(imdb["test"][k]["text"])
        break

critique 4 : vraie étiquette 0, prédiction 1

First off let me say, If you haven't enjoyed a Van Damme movie since bloodsport, you probably will not like this movie. Most of these movies may not have the best plots or best actors but I enjoy these kinds of movies for what they are. This movie is much better than any of the movies the other action guys (Segal and Dolph) have thought about putting out the past few years. Van Damme is good in the movie, the movie is only worth watching to Van Damme fans. It is not as good as Wake of Death (which i highly recommend to anyone of likes Van Damme) or In hell but, in my opinion it's worth watching. It has the same type of feel to it as Nowhere to Run. Good fun stuff!


In [13]:
def voisins(E, mot, k=8):
    """les k plus proches voisins (similarité cosinus) d'un mot dans la matrice d'embeddings E"""
    E = F.normalize(E.detach().cpu(), dim=1)
    sims = E @ E[imdb_vocab[mot]]
    return imdb_vocab.decode(sims.topk(k + 1).indices[1:])

MOTS = ["good", "bad", "boring", "excellent", "funny", "actor"]
for mot in MOTS:
    print(f"{mot:10s} -> {voisins(modeleMoyen.embedding.weight, mot)}")

# visualisation dans le Projector de tensorboard (les 2000 mots les plus fréquents, hors mots spéciaux)
writer = SummaryWriter(TB_PATH / f"projector-{time.strftime('%H%M%S')}")
writer.add_embedding(modeleMoyen.embedding.weight[2:2002].detach().cpu(), metadata=imdb_vocab.itos[2:2002], tag="moyen")
writer.close()

good       -> ['thirsty', 'stood', 'results', 'hardy', 'knowing', 'hickock', 'snake', 'silverstone']
bad        -> ['dull', 'pointless', 'fails', 'no', 'sense', 'annoying', 'rehash', 'worst']
boring     -> ['fake', 'timberlake', 'paquin', 'alicia', 'swear', 'kornbluth', 'acclaimed', 'dreadfully']
excellent  -> ['pleasure', 'lovingly', 'heartwarming', 'knowing', 'gem', 'noir', 'deanna', 'wonderful']
funny      -> ['attorney', 'award', 'dinocroc', 'branagh', 'gerald', 'resume', 'possibilities', 'accurate']
actor      -> ['concentrates', 'haphazard', 'significantly', 'murderer', 'prove', 'receiving', 'safely', 'pieces']


# <a id="sec-6"></a>4. Représentations pré-apprises et fine-tuning

Une option de `nn.Embedding` (`nn.Embedding.from_pretrained`) permet de charger des représentations pré-entraînées. Dans ce cas, par défaut, la couche de représentation est **gelée** : elle n'est pas mise à jour durant l'apprentissage. On peut *fine-tuner* les représentations, c'est-à-dire les adapter à notre tâche, en les dégelant (argument `freeze=False`, ce qui revient à remettre à `True` le flag `requires_grad` du tenseur de représentation).

On utilise les vecteurs [fasttext](https://fasttext.cc/docs/en/pretrained-vectors.html) appris sur le Wikipedia en anglais simplifié (`wiki.simple.vec` : 110 000 mots, dimension 300, ~300 Mo téléchargés une fois dans `data/`). Les mots de notre vocabulaire absents de fasttext (et `<pad>`, `<unk>`) reçoivent le vecteur nul.

Implémentez et testez les deux réseaux : embeddings fasttext gelés, puis fine-tunés.

In [14]:
VEC_PATH = Path("data/wiki.simple.vec")
if not VEC_PATH.exists():
    torch.hub.download_url_to_file("https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.simple.vec", VEC_PATH)

def load_vectors(path, vocab):
    """matrice (len(vocab), dim) des vecteurs du fichier texte (format : 'mot v1 v2 ... vd' par ligne)"""
    with open(path, encoding="utf-8") as f:
        n, dim = map(int, f.readline().split())     # entête : nb de mots, dimension
        vectors = torch.zeros(len(vocab), dim)
        found = 0
        for line in f:
            word, rest = line.rstrip().split(" ", 1)
            if word in vocab:
                vectors[vocab[word]] = torch.tensor([float(v) for v in rest.split(" ")])
                found += 1
    print(f"{found}/{len(vocab)} mots du vocabulaire trouvés dans fasttext")
    return vectors

fast_imdb_vectors = load_vectors(VEC_PATH, imdb_vocab)

12601/13395 mots du vocabulaire trouvés dans fasttext


In [ ]:
## Création du modèle moyen à partir des représentations fasttext, embeddings gelés
##  TODO 

In [ ]:
## Idem avec fine-tuning des embeddings
##  TODO 

**Questions.**
1. Comparer les performances des 3 modèles (dans tensorboard). Quel pourcentage des paramètres a été mis à jour pendant l'apprentissage dans chaque cas ?
2. Comparer les plus proches voisins des mots dans les embeddings appris (partie 3), dans fasttext, et après fine-tuning. Comment l'expliquer ?

In [17]:
for m in [modeleMoyen, modeleMoyenPretrained, modeleMoyenPretrained_fine]:
    n_total = sum(p.numel() for p in m.parameters())
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{m.name:20s} test acc {evaluate(m, imdb_test_dataloader)[1]:.3f} | "
          f"{n_train:9d} paramètres appris / {n_total:9d} ({n_train / n_total:.1%})")

for mot in MOTS:
    print(f"\n{mot}\n   appris    : {voisins(modeleMoyen.embedding.weight, mot)}"
          f"\n   fasttext  : {voisins(fast_imdb_vectors, mot)}"
          f"\n   fine-tuné : {voisins(modeleMoyenPretrained_fine.embedding.weight, mot)}")

moyen                test acc 0.868 |   1345612 paramètres appris /   1345612 (100.0%)


moyen_fasttext       test acc 0.792 |     16112 paramètres appris /   4034612 (0.4%)


moyen_fasttext_fine  test acc 0.870 |   4034612 paramètres appris /   4034612 (100.0%)

good
   appris    : ['thirsty', 'stood', 'results', 'hardy', 'knowing', 'hickock', 'snake', 'silverstone']
   fasttext  : ['excellent', 'reasonable', 'luck', 'bad', 'excellently', 'decent', 'improves', 'helpful']
   fine-tuné : ['excellent', 'improves', 'luck', 'excellently', 'honest', 'plenty', 'wonderful', 'satisfying']

bad
   appris    : ['dull', 'pointless', 'fails', 'no', 'sense', 'annoying', 'rehash', 'worst']
   fasttext  : ['good', 'wrong', 'worse', 'horrible', 'unpleasant', 'badness', 'dreadful', 'badly']
   fine-tuné : ['horrible', 'worse', 'poor', 'pathetically', 'terrible', 'awful', 'clichéd', 'excruciatingly']

boring
   appris    : ['fake', 'timberlake', 'paquin', 'alicia', 'swear', 'kornbluth', 'acclaimed', 'dreadfully']
   fasttext  : ['wasting', 'bored', 'yelling', 'annoying', 'scaring', 'worrying', 'bothering', 'ignoring']
   fine-tuné : ['commenter', 'empathize', 'stunk', 'hobgob

In [ ]:
##  TODO 

# <a id="sec-7"></a>5. Convolutions et texte

Le modèle précédent est relativement pauvre : le texte est représenté uniquement par le barycentre des représentations des tokens, ce qui élimine toute notion de séquentialité (« not good » et « good, not bad » ont presque la même représentation). Une première manière de traiter la séquentialité est d'utiliser des couches convolutionnelles et de pooling, comme en image. Cette fois cependant, la convolution sera en une dimension (le long du texte) et non pas deux : [`nn.Conv1d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html) prend en entrée un tenseur `batch x canaux x longueur`, les canaux étant ici les dimensions de l'embedding.

L'interaction entre l'opérateur de convolution et de max-pooling peut être vue de la manière suivante :

1. la convolution permet de détecter un ensemble de motifs (par exemple « un bon film », « très bon film », « un film déplorable ») sur des séquences courtes, dont la longueur dépend de la taille du noyau (dans cet exemple, 3 mots) ;
2. le max-pooling permet de résumer l'information capturée par les filtres sur une sous-séquence de taille plus grande.

Ces opérations sont répétées plusieurs fois, comme en image, afin de détecter des motifs de taille et de complexité croissantes. Par exemple, on peut détecter « j'ai beaucoup » / « j'ai énormément » / « ai pas beaucoup » et « ce film » / « cet acteur » (4 filtres) sur la première couche de convolution. La seconde couche de convolution va combiner les informations pour détecter un jugement sur un film ou un acteur.

La dernière couche est généralement un maximum global (sur toute la longueur, pour chaque filtre de sortie), suivi d'un classifieur linéaire.

## <a id="sec-8"></a>5.1 Portée (champ réceptif) d'une sortie

Pour étudier ce que fait le CNN (partie 6), il faudra savoir à quelle partie du texte correspond chaque sortie de la dernière couche de convolution. Par exemple, si on considère une convolution avec une taille de noyau 3 et un stride de 1, alors la 1ère sortie correspond au texte entre les positions 1 et 3, la 2ème à celui entre les positions 2 et 4, etc. Si on ajoute un max-pooling (noyau de taille 2, stride 2), alors la 1ère sortie correspond au texte entre les positions 1 et 4, la 2ème aux positions 3 et 6, etc.

Il faut maintenant généraliser. Pour cela, nous allons considérer que la $i$-ème opération (convolution ou pooling) est définie seulement par la taille de son noyau $w_i$ (*kernel width*) et son déplacement $s_i$ (*stride*). Nous nous intéressons aux deux valeurs qui caractérisent l'ensemble des transformations jusqu'à l'opération $i$ :

* la longueur $W_i$ des entrées correspondant à une sortie,
* le déplacement $S_i$ dans les entrées correspondant à un déplacement unitaire dans les sorties.

(avec $W_0 = S_0 = 1$ : avant toute opération, une sortie = un token).

**Questions.**
1. Donner la formule de récurrence qui permet, étant donnés $W_i$, $S_i$, de déterminer $W_{i+1}$ et $S_{i+1}$ sachant $w_{i+1}$ et $s_{i+1}$.
2. Soit $(y_0,\ldots, y_{L-1})$ la sortie du CNN. Donner la formule qui, étant donnée la position $j$ de la sortie $y_j$, permet de déterminer les indices correspondants dans la séquence d'entrée.
3. Implémenter la méthode `__call__(width, stride)` de la classe `Scope`, qui met à jour $(W, S)$ quand on ajoute une opération de noyau `width` et de pas `stride`. Vérifier sur les deux exemples ci-dessus.

In [ ]:
class Scope:
    """Calcule la portée (W = width) et le déplacement (S = stride) d'une sortie après une suite d'opérations"""
    def __init__(self):
        self.width = 1
        self.stride = 1

    def __call__(self, width, stride=1):
        ##  TODO 
        return self

    def __repr__(self):
        return f"(W={self.width}, S={self.stride})"

print(Scope()(3, 1))            # conv k=3, s=1            : attendu (W=3, S=1)
print(Scope()(3, 1)(2, 2))      # puis max-pooling k=2, s=2 : attendu (W=4, S=2)

## <a id="sec-9"></a>5.2 Le modèle convolutionnel

L'architecture est fournie : 3 convolutions de noyau 3, séparées par des max-pooling de noyau 2 et de pas 2 (qui divisent la longueur par 2), puis un maximum global sur la longueur et une couche linéaire. Les embeddings sont initialisés avec fasttext (gelés ou fine-tunés).

Quelle est la portée d'une sortie de la dernière convolution ? Vérifier avec la valeur affichée.

In [20]:
class ModelCNN(nn.Module):
    def __init__(self, vocab_size, input_dim, output_dim, nfeat, padding_idx, pre_trained=None, freeze=True, name="cnn"):
        super().__init__()
        self.name = name
        self.nfeat = nfeat
        if pre_trained is None:
            self.embedding = nn.Embedding(vocab_size, input_dim, padding_idx=padding_idx)
        else:
            # clone() : from_pretrained ne copie PAS le tenseur ; sans clone, le fine-tuning modifierait en place
            # pre_trained (fast_imdb_vectors), et donc les embeddings de tous les modèles qui le partagent !
            self.embedding = nn.Embedding.from_pretrained(pre_trained.clone(), freeze=freeze, padding_idx=padding_idx)

        self.convolutions = nn.Sequential(
            nn.Conv1d(input_dim, 100, 3),   # canaux en entrée, canaux en sortie (= nb de filtres), taille du noyau
            nn.MaxPool1d(2, 2),             # noyau, stride : la longueur est divisée par 2
            nn.ReLU(),
            nn.Conv1d(100, 100, 3),
            nn.MaxPool1d(2, 2),
            nn.ReLU(),
            nn.Conv1d(100, self.nfeat, 3),
        )

        self.scope = Scope()
        for m in self.convolutions:
            if isinstance(m, nn.Conv1d):
                self.scope = self.scope(m.kernel_size[0], m.stride[0])
            elif isinstance(m, nn.MaxPool1d):
                self.scope = self.scope(m.kernel_size, m.stride)

        self.fc = nn.Linear(self.nfeat, output_dim)

    def convolution_fn(self, x):
        """sorties de la dernière convolution : batch x nfeat x longueur"""
        x = self.embedding(x).transpose(1, 2)    # batch x longueur x dim -> batch x dim x longueur
        return self.convolutions(x)

    def forward(self, x):
        y = self.convolution_fn(x)
        return self.fc(y.max(2)[0])              # maximum global sur la longueur

print("portée d'une sortie :", ModelCNN(10, 300, 2, 20, PAD).scope)

portée d'une sortie : (W=18, S=4)


In [ ]:
##  TODO 

Apprentissage des deux variantes (embeddings gelés / fine-tunés). Chaque apprentissage prend quelques minutes (cf. commentaires en début de notebook) : les modèles sont sauvegardés dans `models/`.

In [22]:
modelCNN = ModelCNN(len(imdb_vocab), 300, 2, 20, PAD, fast_imdb_vectors, name="cnn_fasttext")
modelCNN = train_or_load(modelCNN, EPOCHS, "imdb_cnn_fasttext.pt")

modelCNN_fine = ModelCNN(len(imdb_vocab), 300, 2, 20, PAD, fast_imdb_vectors, freeze=False, name="cnn_fasttext_fine")
modelCNN_fine = train_or_load(modelCNN_fine, EPOCHS, "imdb_cnn_fasttext_fine.pt")

for phr in ["this movie is awful", "the movie is great", "not bad at all", "I expected a great movie, it was boring"]:
    print(f"{predict(modelCNN_fine, phr):.2f}  {phr}")

[cnn_fasttext] chargé depuis models/imdb_cnn_fasttext.pt | test acc 0.865


[cnn_fasttext_fine] chargé depuis models/imdb_cnn_fasttext_fine.pt | test acc 0.878
0.00  this movie is awful
0.99  the movie is great
0.11  not bad at all
0.01  I expected a great movie, it was boring


# <a id="sec-10"></a>6. Que détectent les filtres du CNN ?

Afin d'étudier ce que fait le CNN, nous allons nous intéresser à la dernière couche avant le maximum global ; plus particulièrement, nous allons chercher les sous-séquences (dans le jeu d'apprentissage) qui activent le plus chaque filtre de sortie.

Grâce à la portée calculée en 5.1, la sortie $y_j$ d'un filtre correspond aux tokens `x[j * S : j * S + W]`. Le code ci-dessous (fourni) parcourt les données d'apprentissage, et garde pour chaque filtre les $K$ sous-séquences qui l'activent le plus (avec un tas, module `heapq`).

**Question.** Les filtres sont-ils interprétables ? Comment sont-ils utilisés par la couche linéaire finale (regarder le signe des poids `modelCNN_fine.fc.weight`) ?

In [23]:
class Sample:
    """une sous-séquence de texte et son activation (comparable par activation, pour heapq)"""
    def __init__(self, value, text):
        self.value = value
        self.text = text

    def __lt__(self, other):
        return self.value < other.value

K = 10
model = modelCNN_fine
W, S = model.scope.width, model.scope.stride
topk = [[] for _ in range(model.nfeat)]

model.eval()
with torch.no_grad():
    for x, _ in tqdm(imdb_train_dataloader, desc="analyse des filtres"):
        y = model.convolution_fn(x.to(next(model.parameters()).device)).cpu()     # batch x nfeat x lmax
        bs, nfeat, lmax = y.shape
        # pour chaque filtre, les K plus fortes activations du batch (toutes positions confondues)
        yt = y.transpose(0, 1).reshape(nfeat, -1)       # nfeat x (batch * lmax)
        values, indices = yt.topk(K, dim=1)
        for i in range(nfeat):
            for v, ind in zip(values[i].tolist(), indices[i].tolist()):
                batch_ix, pos = ind // lmax, ind % lmax   # on retrouve l'exemple et la position
                sample = Sample(v, x[batch_ix, pos * S: pos * S + W])
                if len(topk[i]) < K:
                    heapq.heappush(topk[i], sample)
                elif topk[i][0].value < v:
                    heapq.heapreplace(topk[i], sample)

w_fc = model.fc.weight.detach().cpu()    # 2 x nfeat : poids vers les classes négatif / positif
for i in range(model.nfeat):
    print(f"==== Filtre {i} (poids positif - négatif : {w_fc[1, i] - w_fc[0, i]:+.2f}) ====")
    for sample in sorted(topk[i], reverse=True)[:5]:
        print(f"{sample.value:6.2f} : {' '.join(imdb_vocab.decode(sample.text))}")

==== Filtre 0 (poids positif - négatif : +0.36) ====
 18.49 : and the acting is also great , especially jon voight and burt reynolds . magnificent acting in this
 17.82 : memorable documentary . both big edie and little edie are unforgettable and their utter lack of self -
 17.73 : the dialogue is wonderfully witty and refreshing and the atmosphere enchanting . ralph richardson was a delight to
 17.68 : them . i just truly appreciated this film as it beautifully portrays the impact american indulgence has over
 17.62 : the acting is really good . jake gyllenhaal delivers a great performance as homer <unk> and chris cooper
==== Filtre 1 (poids positif - négatif : -0.32) ====
 16.09 : music mix worst editing worst script worst all - time directing worst acting worst choreography worst cinematography worst
 15.24 : worst script worst all - time directing worst acting worst choreography worst cinematography worst props worst sets worst
 14.55 : - time directing worst acting worst choreogra

In [ ]:
##  TODO 